# EcoVision Beetle Classifier
Run all cells to launch the classifier UI. Make sure Anaconda is installed and you are running this notebook with the base Python kernel.

In [1]:
import os
import json
import numpy as np
import pandas as pd
import tkinter as tk
from tkinter import ttk, filedialog, messagebox
import tensorflow as tf
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.resnet50 import preprocess_input
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from sklearn.metrics import confusion_matrix

MODELS_DIR = r"C:\Users\timl9\OneDrive\Desktop\EcoVision\Models"

def load_model_list():
    return [d for d in os.listdir(MODELS_DIR) if os.path.isdir(os.path.join(MODELS_DIR, d))]

def load_model(model_name):
    model_dir = os.path.join(MODELS_DIR, model_name)
    weights = [f for f in os.listdir(model_dir) if f.endswith('.h5')]
    if not weights:
        raise FileNotFoundError('No .h5 file found in model folder')
    model = tf.keras.models.load_model(os.path.join(model_dir, weights[0]))
    info_path = os.path.join(model_dir, 'model_info.json')
    info = json.load(open(info_path)) if os.path.exists(info_path) else {}
    cm_path = os.path.join(model_dir, 'confusion_matrix.csv')
    cm = pd.read_csv(cm_path, index_col=0) if os.path.exists(cm_path) else None
    return model, info, cm

def predict_single(model, class_names, img_path):
    img = image.load_img(img_path, target_size=(224, 224))
    arr = preprocess_input(np.expand_dims(image.img_to_array(img), axis=0))
    pred = model.predict(arr, verbose=0)[0]
    return class_names[np.argmax(pred)], float(np.max(pred)), pred

def predict_folder(model, class_names, folder_path):
    results = []
    true_labels = []
    pred_labels = []
    subfolders = [d for d in os.listdir(folder_path) if os.path.isdir(os.path.join(folder_path, d))]
    for cls in subfolders:
        cls_dir = os.path.join(folder_path, cls)
        for fname in os.listdir(cls_dir):
            if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
            fpath = os.path.join(cls_dir, fname)
            predicted, confidence, _ = predict_single(model, class_names, fpath)
            results.append({'File': fname, 'True Label': cls, 'Predicted': predicted, 'Confidence': f'{confidence:.1%}'})
            true_labels.append(cls)
            pred_labels.append(predicted)
    return results, true_labels, pred_labels

print('Libraries loaded successfully')

Libraries loaded successfully


In [2]:
def plot_confusion_matrix(cm_df, parent_frame):
    fig, ax = plt.subplots(figsize=(4, 3))
    data = cm_df.values.astype(float)
    im = ax.imshow(data, cmap='Blues')
    ax.set_xticks(range(len(cm_df.columns)))
    ax.set_yticks(range(len(cm_df.index)))
    ax.set_xticklabels(cm_df.columns, rotation=45, ha='right', fontsize=7)
    ax.set_yticklabels(cm_df.index, fontsize=7)
    for i in range(len(cm_df.index)):
        for j in range(len(cm_df.columns)):
            ax.text(j, i, int(data[i, j]), ha='center', va='center', fontsize=7,
                    color='white' if data[i, j] > data.max() / 2 else 'black')
    ax.set_xlabel('Predicted', fontsize=8)
    ax.set_ylabel('True', fontsize=8)
    fig.tight_layout()
    canvas = FigureCanvasTkAgg(fig, master=parent_frame)
    canvas.draw()
    canvas.get_tk_widget().pack(fill='both', expand=True)
    plt.close(fig)

def build_ui():
    root = tk.Tk()
    root.title('EcoVision Beetle Classifier')
    root.geometry('900x700')

    state = {'model': None, 'class_names': [], 'model_name': ''}

    # ── Top: model selector ──────────────────────────────────────────
    top = tk.Frame(root, pady=8)
    top.pack(fill='x', padx=10)
    tk.Label(top, text='Select Model:', font=('Arial', 11)).pack(side='left')
    model_var = tk.StringVar()
    model_dropdown = ttk.Combobox(top, textvariable=model_var, values=load_model_list(), width=35, state='readonly')
    model_dropdown.pack(side='left', padx=8)
    status_label = tk.Label(top, text='', font=('Arial', 10), fg='green')
    status_label.pack(side='left', padx=8)

    # ── Middle: info panel (left) + tabs (right) ─────────────────────
    middle = tk.Frame(root)
    middle.pack(fill='both', expand=True, padx=10, pady=5)

    info_frame = tk.LabelFrame(middle, text='Model Info', width=280, font=('Arial', 10))
    info_frame.pack(side='left', fill='y', padx=(0, 8))
    info_frame.pack_propagate(False)

    info_text = tk.Text(info_frame, width=30, font=('Arial', 9), state='disabled', wrap='word')
    info_text.pack(fill='x', padx=5, pady=5)
    cm_frame = tk.Frame(info_frame)
    cm_frame.pack(fill='both', expand=True)

    right_frame = tk.Frame(middle)
    right_frame.pack(side='left', fill='both', expand=True)

    notebook = ttk.Notebook(right_frame)
    notebook.pack(fill='both', expand=True)

    # ── Single image tab ─────────────────────────────────────────────
    single_tab = tk.Frame(notebook, pady=10)
    notebook.add(single_tab, text='Single Image')

    tk.Button(single_tab, text='Load Image', font=('Arial', 11),
              command=lambda: run_single()).pack(pady=10)
    single_result = tk.Label(single_tab, text='', font=('Arial', 14))
    single_result.pack(pady=5)
    conf_frame = tk.Frame(single_tab)
    conf_frame.pack(fill='x', padx=20)

    # ── Folder tab ───────────────────────────────────────────────────
    folder_tab = tk.Frame(notebook, pady=10)
    notebook.add(folder_tab, text='Folder of Images')

    tk.Button(folder_tab, text='Select Folder', font=('Arial', 11),
              command=lambda: run_folder()).pack(pady=10)

    table_frame = tk.Frame(folder_tab)
    table_frame.pack(fill='both', expand=True, padx=5)
    cols = ('File', 'True Label', 'Predicted', 'Confidence')
    tree = ttk.Treeview(table_frame, columns=cols, show='headings', height=10)
    for col in cols:
        tree.heading(col, text=col)
        tree.column(col, width=140)
    scrollbar = ttk.Scrollbar(table_frame, orient='vertical', command=tree.yview)
    tree.configure(yscrollcommand=scrollbar.set)
    tree.pack(side='left', fill='both', expand=True)
    scrollbar.pack(side='right', fill='y')

    folder_cm_frame = tk.LabelFrame(folder_tab, text='Confusion Matrix')
    folder_cm_frame.pack(fill='both', expand=True, padx=5, pady=5)

    # ── Load model ───────────────────────────────────────────────────
    def on_model_select(event=None):
        name = model_var.get()
        if not name:
            return
        status_label.config(text='Loading...', fg='orange')
        root.update()
        try:
            model, info, cm = load_model(name)
            state['model'] = model
            state['model_name'] = name
            state['class_names'] = list(info.get('training_samples', {}).keys()) if info else []
            status_label.config(text='Model loaded', fg='green')

            # Update info panel
            info_text.config(state='normal')
            info_text.delete('1.0', 'end')
            if info:
                info_text.insert('end', f"Model: {info.get('model', 'N/A')}\n")
                info_text.insert('end', f"Dataset: {info.get('dataset', 'N/A')}\n\n")
                info_text.insert('end', 'Training samples:\n')
                for cls, count in info.get('training_samples', {}).items():
                    info_text.insert('end', f'  {cls}: {count}\n')
            info_text.config(state='disabled')

            # Plot confusion matrix
            for widget in cm_frame.winfo_children():
                widget.destroy()
            if cm is not None:
                plot_confusion_matrix(cm, cm_frame)
        except Exception as e:
            status_label.config(text=f'Error: {e}', fg='red')

    model_dropdown.bind('<<ComboboxSelected>>', on_model_select)

    # ── Single image prediction ───────────────────────────────────────
    def run_single():
        if not state['model']:
            messagebox.showwarning('No model', 'Please select a model first.')
            return
        path = filedialog.askopenfilename(
            initialdir=r'C:\Users\timl9\OneDrive\Desktop\EcoVision',
            filetypes=[('Images', '*.jpg *.jpeg *.png')])
        if not path:
            return
        predicted, confidence, all_preds = predict_single(state['model'], state['class_names'], path)
        single_result.config(text=f'{predicted}  ({confidence:.1%} confidence)')

        for widget in conf_frame.winfo_children():
            widget.destroy()
        for cls, prob in sorted(zip(state['class_names'], all_preds), key=lambda x: -x[1]):
            row = tk.Frame(conf_frame)
            row.pack(fill='x', pady=1)
            tk.Label(row, text=f'{cls}', width=20, anchor='w', font=('Arial', 9)).pack(side='left')
            bar = tk.Canvas(row, height=16, width=int(prob * 200), bg='steelblue')
            bar.pack(side='left')
            tk.Label(row, text=f'{prob:.1%}', font=('Arial', 9)).pack(side='left', padx=4)

    # ── Folder prediction ─────────────────────────────────────────────
    def run_folder():
        if not state['model']:
            messagebox.showwarning('No model', 'Please select a model first.')
            return
        folder = filedialog.askdirectory(initialdir=r'C:\Users\timl9\OneDrive\Desktop\EcoVision')
        if not folder:
            return
        status_label.config(text='Running predictions...', fg='orange')
        root.update()
        results, true_labels, pred_labels = predict_folder(state['model'], state['class_names'], folder)

        for row in tree.get_children():
            tree.delete(row)
        for r in results:
            tree.insert('', 'end', values=(r['File'], r['True Label'], r['Predicted'], r['Confidence']))

        for widget in folder_cm_frame.winfo_children():
            widget.destroy()
        if true_labels:
            all_classes = sorted(set(true_labels + pred_labels))
            cm = confusion_matrix(true_labels, pred_labels, labels=all_classes)
            cm_df = pd.DataFrame(cm, index=all_classes, columns=all_classes)
            plot_confusion_matrix(cm_df, folder_cm_frame)

        status_label.config(text=f'Done — {len(results)} images', fg='green')

    root.mainloop()

build_ui()